# Evaluation

## Objective
Evaluate the conversational recommendation pipeline honestly and at the right level of ambition for this project.

## Inputs
- retrieval artifacts from notebook 02
- curated scenarios from `../data/curated/scenario_suite.json`
- shared helpers in `functions/`
- raw `items.parquet` and `reviews.parquet` accessed lazily via DuckDB where grounding checks require them

## Outputs
- scoped retrieval checks
- parser comparison tables on curated requests
- reranking behavior checks on representative scenarios
- grounding quality rubrics and failure analysis

## What Is Being Evaluated And Why
This notebook does not assume benchmark-grade labels for every stage. Instead, it evaluates what can be checked today: whether retrieval is coherent, whether the parser extracts the intended structure, whether reranking respects the intended policy, and whether grounding evidence is relevant enough to support a careful response.


## Evaluation Goals

The right question is not “can we manufacture one grand metric?” The right question is whether each stage behaves credibly within the actual scope of the project. That means mixing lightweight quantitative checks with curated qualitative inspection where ground truth is necessarily partial.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

PROJECT_ROOT_CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    if (candidate / "functions").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root containing functions/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from functions.evaluation import grounding_rubric, keyword_hit_count, normalize_list
from functions.grounding import assemble_grounding_evidence
from functions.io import (
    connect_catalog_views,
    load_cross_encoder,
    load_curated_scenarios,
    load_retrieval_artifacts,
    materialize_scenarios,
    resolve_reference_strategy,
)
from functions.parsing import build_parser_catalog, parse_user_request
from functions.reranking import apply_hard_constraints, rerank_candidates
from functions.retrieval import hybrid_search, search_similar

artifacts = load_retrieval_artifacts(load_encoder_model=True)
candidate_pool = artifacts["candidate_pool"]
parser_catalog = build_parser_catalog(candidate_pool)
scenario_suite = load_curated_scenarios()
demo_scenarios = materialize_scenarios(scenario_suite["demo_scenarios"], candidate_pool)
con = connect_catalog_views(include_reviews=True)
cross_encoder = load_cross_encoder()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Retrieval Evaluation

Retrieval is only worth keeping as the first stage if it stays coherent at the level of product type and rough use case. The checks below are lightweight: they ask whether the early candidates contain the right kind of item, not whether retrieval has solved recommendation on its own.

This block evaluates two retrieval settings:

- **text_query**: direct search from a short query such as `wireless bluetooth headphones with noise cancelling`, `nonstick frying pan for induction cooking`, and `adjustable dumbbells for home gym workouts`
- **item_to_item**: similarity search starting from a resolved anchor item in the same domain, one for headphones, one for cookware, and one for dumbbells

The table should be read as follows:

- `label` identifies the scenario being tested
- `mode` tells you whether the test starts from raw query text or from a reference item
- `hits_at_5` counts how many of the top 5 retrieved items contain the expected domain keywords
- `min_hits_at_5` is the minimum threshold required for that scenario to count as a pass


In [2]:
retrieval_rows = []
for spec in scenario_suite["retrieval_eval_queries"]:
    results = hybrid_search(
        spec["query"],
        artifacts,
        top_k=5,
        source_filter=spec["source_filter"],
    )
    hits = keyword_hit_count(results, spec["expected_keywords"])
    retrieval_rows.append(
        {
            "label": spec["label"],
            "mode": "text_query",
            "hits_at_5": hits,
            "min_hits_at_5": spec["min_hits_at_5"],
            "passes": hits >= spec["min_hits_at_5"],
        }
    )

for spec in scenario_suite["anchor_eval_strategies"]:
    reference_item = resolve_reference_strategy(spec["reference_strategy"], candidate_pool)
    results = search_similar(
        reference_item["resolved_parent_asin"],
        artifacts,
        top_k=5,
        source_filter=spec["source_filter"],
    )
    hits = keyword_hit_count(results, spec["expected_keywords"])
    retrieval_rows.append(
        {
            "label": spec["label"],
            "mode": "item_to_item",
            "hits_at_5": hits,
            "min_hits_at_5": spec["min_hits_at_5"],
            "passes": hits >= spec["min_hits_at_5"],
        }
    )

retrieval_eval = pd.DataFrame(retrieval_rows)
display(retrieval_eval)

,label,mode,hits_at_5,min_hits_at_5,passes
0,headphones_query,text_query,5,3,True
1,cookware_query,text_query,5,3,True
2,dumbbells_query,text_query,5,3,True
3,headphones_anchor,item_to_item,5,3,True
4,cookware_anchor,item_to_item,5,3,True
5,dumbbells_anchor,item_to_item,2,3,False


## Parser Evaluation

The parser is evaluated against expected field patterns in the curated scenario suite. This is still a small hand-built set, but it is enough to test whether the parser extracts the kinds of structured signals the downstream pipeline actually uses.

The six scenarios here cover the main request types used throughout the project:

- a budget-constrained headphones request with a brand exclusion
- a cheaper-than-reference headphones request that depends on prior item context
- an induction frying pan request with a clear use case
- a dumbbells request for home gym use
- a lightweight laptop request for programming
- an intentionally ambiguous gift request that should trigger clarification

Each row checks whether the parser recovered the expected internal structure for that scenario. In other words, this table is not evaluating recommendation quality yet. It is checking whether the parser produced the right contract for later retrieval, reranking, grounding, and clarification behavior.


In [3]:
parser_rows = []
for scenario in demo_scenarios:
    expected = scenario["expected_parser"]
    parsed = parse_user_request(
        scenario["query"],
        parser_catalog,
        reference_item_context=scenario["reference_item_context"],
    )
    actual_terms = normalize_list(parsed["hard_constraints"]["must_include_terms"])
    expected_terms = normalize_list(expected.get("must_include_terms", []))
    actual_brands = normalize_list(parsed["excluded_brands"])
    expected_brands = normalize_list(expected.get("excluded_brands", []))
    actual_soft = normalize_list(parsed["soft_preferences"])
    expected_soft = normalize_list(expected.get("soft_preferences", []))
    actual_grounding = normalize_list(parsed["grounding_needs"])
    expected_grounding = normalize_list(expected.get("grounding_needs", []))
    reference_present = any(item.get("resolved_parent_asin") for item in parsed["reference_items"])

    parser_rows.append(
        {
            "label": scenario["label"],
            "intent_match": parsed["user_intent"] == expected.get("user_intent"),
            "domain_match": parsed["domain_hint"] == expected.get("domain_hint"),
            "must_include_match": set(expected_terms).issubset(set(actual_terms)),
            "brand_match": set(expected_brands).issubset(set(actual_brands)),
            "soft_pref_match": set(expected_soft).issubset(set(actual_soft)),
            "grounding_match": set(expected_grounding).issubset(set(actual_grounding)),
            "clarification_match": parsed["clarification_needed"] == expected.get("clarification_needed"),
            "reference_match": reference_present == expected.get("expects_reference", False),
        }
    )

parser_eval = pd.DataFrame(parser_rows)
parser_eval["all_checks_pass"] = parser_eval.drop(columns=["label"]).all(axis=1)
display(parser_eval)

,label,intent_match,domain_match,must_include_match,brand_match,soft_pref_match,grounding_match,clarification_match,reference_match,all_checks_pass
0,budget_headphones_avoid_beats,True,True,True,True,True,True,True,True,True
1,reference_cheaper_headphones,True,True,True,True,True,True,True,True,True
2,induction_frying_pan,True,True,True,True,True,True,True,True,True
3,dumbbells_home_gym,True,True,True,True,True,True,True,True,True
4,lightweight_programming_laptop,True,True,True,True,True,True,True,True,True
5,gift_ambiguity,True,True,True,True,True,True,True,True,True


## Reranking Evaluation

Reranking should do something specific: respect the policy implied by the parsed request. The checks below ask whether the finalists survive the hard constraints and whether reranking still produces a non-empty, coherent final set on representative scenarios.

This section focuses on four scenarios where explicit ranking policy matters most:

- budget headphones with a Beats exclusion
- cheaper-than-reference headphones, where the user points back to an earlier item
- induction frying pans, where use-case fit matters more than a vague text match
- lightweight laptops for programming, where the request is clear but still needs ranking trade-offs

The columns show how the candidate set changes as the pipeline gets stricter:

- `baseline_count` is the broad retrieval pool before constraints
- `filtered_count` is the number of candidates that survive the hard filters
- `finalist_count` is the final reranked shortlist
- `all_finalists_pass_constraints` checks whether every returned finalist still respects the requested policy


In [4]:
rerank_rows = []
rerank_labels = [
    "budget_headphones_avoid_beats",
    "reference_cheaper_headphones",
    "induction_frying_pan",
    "lightweight_programming_laptop",
]
for scenario in demo_scenarios:
    if scenario["label"] not in rerank_labels:
        continue
    parsed = parse_user_request(
        scenario["query"],
        parser_catalog,
        reference_item_context=scenario["reference_item_context"],
    )
    result = rerank_candidates(parsed, artifacts, top_k_retrieval=20, top_k_final=5)
    finalists = result["reranked_candidates"]
    reference = next((item for item in parsed["reference_items"] if item.get("resolved_parent_asin")), None)
    reference_price = reference.get("price") if reference else None
    keep_mask = apply_hard_constraints(finalists, parsed, reference_price=reference_price)
    rerank_rows.append(
        {
            "label": scenario["label"],
            "baseline_count": len(result["baseline_candidates"]),
            "filtered_count": len(result["filtered_candidates"]),
            "finalist_count": len(finalists),
            "all_finalists_pass_constraints": bool(keep_mask.all()) if len(finalists) else False,
            "top_keyword_hits": (
                keyword_hit_count(finalists.head(3), scenario["retrieval_keywords"])
                if scenario["retrieval_keywords"]
                else None
            ),
        }
    )

rerank_eval = pd.DataFrame(rerank_rows)
display(rerank_eval)

,label,baseline_count,filtered_count,finalist_count,all_finalists_pass_constraints,top_keyword_hits
0,budget_headphones_avoid_beats,20,13,5,True,3
1,reference_cheaper_headphones,20,7,5,True,3
2,induction_frying_pan,20,18,5,True,3
3,lightweight_programming_laptop,20,20,5,True,3


## Grounding / Explanation Evaluation

Grounding quality is harder to evaluate numerically without human labels, so this notebook uses a small rubric. The rubric checks whether the evidence seems relevant, whether it looks supportive enough to justify a response, whether it is specific rather than generic, and how much risk of overclaiming remains.

The grounding checks reuse the same four representative scenarios from the reranking section. That makes the context easier to compare: first a scenario produces finalists, then this section asks whether the attached evidence is strong enough to support those finalists in a careful explanation.

The table should be read as a compact explanation-quality summary:

- `evidence_rows` is the amount of evidence attached across the evaluated finalists
- `relevance` asks whether the evidence overlaps with what the user actually asked for
- `supportiveness` asks whether the evidence could reasonably justify the recommendation
- `specificity` asks whether the evidence is concrete rather than generic boilerplate
- `overclaim_risk` estimates how risky it would be to present the final explanation with confidence


In [5]:
grounding_rows = []
for scenario in demo_scenarios:
    if scenario["label"] not in rerank_labels:
        continue
    parsed = parse_user_request(
        scenario["query"],
        parser_catalog,
        reference_item_context=scenario["reference_item_context"],
    )
    result = rerank_candidates(parsed, artifacts, top_k_retrieval=20, top_k_final=3)
    evidence = assemble_grounding_evidence(
        parsed,
        result["reranked_candidates"],
        con,
        candidate_pool,
        cross_encoder,
        top_review_evidence_per_item=1,
    )
    rubric = grounding_rubric(parsed, result["reranked_candidates"], evidence)
    grounding_rows.append(
        {
            "label": scenario["label"],
            "evidence_rows": len(evidence),
            **rubric,
        }
    )

grounding_eval = pd.DataFrame(grounding_rows)
display(grounding_eval)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,label,evidence_rows,relevance,supportiveness,specificity,overclaim_risk
0,budget_headphones_avoid_beats,6,strong,strong,strong,low
1,reference_cheaper_headphones,6,strong,strong,strong,low
2,induction_frying_pan,6,strong,strong,strong,low
3,lightweight_programming_laptop,6,strong,strong,strong,low


## Failure Analysis And Final Discussion

A few patterns should stay explicit:

- Retrieval is now coherent enough to act as a real candidate generator, but it is still lexical and therefore limited.
- The parser is useful because it is stable and inspectable, not because it fully understands language.
- Reranking is strongest when the request maps cleanly onto explicit metadata fields.
- Grounding helps make the system more defensible, but evidence quality remains uneven.

That is a solid outcome for the current project stage. The notebook sequence is now complete, and the evaluation remains scoped tightly enough to stay believable.
